# sweets example 2 — OPERA CSLC workflow

End-to-end InSAR run over the same LA AOI, this time using pre-geocoded [OPERA L2 CSLC-S1 HDF5s](https://www.jpl.nasa.gov/go/opera/products/cslc-product) from ASF DAAC instead of raw S1 bursts. sweets skips burst2safe and COMPASS entirely on this path — the CSLCs are already at OPERA's 5×10 m posting — and pulls the matching CSLC-STATIC layers for geometry stitching.

**Why use it?** Faster than `--source safe` whenever OPERA has produced for your AOI (mostly CONUS). No COMPASS runtime, no burst database, no orbit files, no DEM download for COMPASS — just download, stitch geometry, dolphin.

**AOI / time**: LA polygon, 2025-12-01 -> 2025-12-31, same as the Sentinel-1 and NISAR example notebooks. ASF has OPERA CSLCs for descending track 71 in this window.

**Expected wall time**: ~3-6 minutes, dominated by the CSLC downloads.

## Earthdata credentials

sweets downloads Sentinel-1 bursts, OPERA CSLCs, NISAR GSLCs and tropo data from NASA Earthdata endpoints. Every subsystem used below (`burst2safe`, `opera-utils`, `asf_search`) expects a `~/.netrc` entry like:

```
machine urs.earthdata.nasa.gov
  login YOUR_EARTHDATA_USERNAME
  password YOUR_EARTHDATA_PASSWORD
```

If you don't have credentials yet, register for free at <https://urs.earthdata.nasa.gov/users/new>.

## Configure the run

Pass `--source opera-cslc`. `--track` is optional on this path — ASF will filter on the AOI alone — but we pin track 71 to keep the result deterministic. Add `--do-tropo` to run the OPERA L4 TROPO-ZENITH correction step after dolphin; it's off here for simplicity.

In [ ]:
from pathlib import Path

WORK_DIR = Path("./example_opera_cslc").resolve()
WORK_DIR.mkdir(exist_ok=True)
CONFIG_YAML = WORK_DIR / "sweets_config.yaml"
print(WORK_DIR)

In [ ]:
!sweets config \
  --bbox=-118.3957 33.7284 -118.3459 33.772 \
  --start 2025-12-01 --end 2025-12-31 \
  --source opera-cslc \
  --track 71 \
  --out-dir {WORK_DIR}/data \
  --work-dir {WORK_DIR} \
  --output {CONFIG_YAML}

In [ ]:
!head -60 {CONFIG_YAML}

## Run the workflow

Step 1: parallel CSLC + CSLC-STATIC + DEM + water mask download. Step 2: stitch the static layers into burst-aligned geometry rasters under `geometry/`. Step 3: dolphin.

(Sweets auto-skips COMPASS, burst2safe and orbit download on the OPERA path.)

In [ ]:
!sweets run {CONFIG_YAML}

## Inspect the outputs

Same dolphin output layout as the burst-subset example:

In [ ]:
!ls {WORK_DIR}/dolphin/

In [ ]:
!ls {WORK_DIR}/dolphin/timeseries/

In [ ]:
!gdalinfo -stats {WORK_DIR}/dolphin/timeseries/velocity.tif | grep -E 'Size|Pixel Size|Unit Type|Minimum|Maximum|StdDev'